# Day 6 — Build GPT from Scratch I (nanoGPT-style)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day06-gpt-from-scratch-i.ipynb)

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
# Expected: installs quietly, no output above this cell

## 1. Tiny Shakespeare: char-level vocab + data loader
V = 65 chars. Targets are inputs shifted by one. Train/val = 90/10.

In [ ]:
import torch, urllib.request, math

# --- download ---
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
try:
    text = open("/tmp/input.txt").read()
except FileNotFoundError:
    urllib.request.urlretrieve(url, "/tmp/input.txt")
    text = open("/tmp/input.txt").read()

# --- vocab ---
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] for i in l)
data = torch.tensor(encode(text), dtype=torch.long)
print("vocab size:", len(chars))           # Expected: 65
print("data tokens:", len(data))           # Expected: ~1115394

# --- train/val + batching ---
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
GPU = torch.cuda.is_available()
batch_size = 64 if GPU else 16
block_size = 128
device = "cuda" if GPU else "cpu"
print(f"device: {device}, batch: {batch_size}, block: {block_size}")

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("xb", xb.shape, "yb", yb.shape)     # Expected: (B, 128) each


## 2. The whole model: embeddings, positions, N blocks, norm, head
Your Day 5 decoder block goes straight in. ~150 lines total.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))
        self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight + self.bias

class CausalMHA(nn.Module):
    def __init__(self, d, n_head):
        super().__init__()
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.n_head = n_head
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=-1)
        q = q.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        return self.proj(F.scaled_dot_product_attention(q, k, v, is_causal=True)
                         .transpose(1, 2).contiguous().view(B, T, D))

class SwiGLU(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.gate, self.up, self.down = nn.Linear(d, d_ff), nn.Linear(d, d_ff), nn.Linear(d_ff, d)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class DecoderBlock(nn.Module):
    def __init__(self, d, n_head):
        super().__init__()
        self.ln1, self.ln2 = RMSNorm(d), RMSNorm(d)
        self.attn = CausalMHA(d, n_head)
        self.mlp = SwiGLU(d, 4 * d)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d, block_size):
        super().__init__()
        self.block_size = block_size
        self.wte = nn.Embedding(vocab, d)          # token embedding
        self.wpe = nn.Embedding(block_size, d)    # learned positions
        self.blocks = nn.Sequential(*[DecoderBlock(d, n_head) for _ in range(n_layer)])
        self.ln_f = RMSNorm(d)                    # final norm
        self.lm_head = nn.Linear(d, vocab, bias=False)   # back to logits
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.wte(idx) + self.wpe(torch.arange(T, device=idx.device))
        logits = self.lm_head(self.ln_f(self.blocks(x)))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

torch.manual_seed(1337)
if GPU:
    cfg = dict(vocab=65, n_layer=6, n_head=6, d=384, block_size=block_size)   # 10.7M
else:
    cfg = dict(vocab=65, n_layer=4, n_head=4, d=256, block_size=block_size)   # ~3.2M
model = GPT(**cfg).to(device)
total = sum(p.numel() for p in model.parameters())
print(f"total params: {total:,}")
# Expected T4:  10,725,888 (~10.7M) | Expected CPU: ~3.2M

# Smoke test: untrained loss must equal ln(65) — uniform guessing
model.eval()
with torch.no_grad():
    _, l0 = model(xb, yb)
print(f"step-0 loss: {l0.item():.3f}  (ln 65 = {math.log(65):.3f})")
# Expected: step-0 loss ≈ 4.17


## 3. Train: log loss every 100 steps, save the curve
T4: 5k steps (~10–20 min wall). CPU: 1k steps, smaller config (~10–20 min wall).

In [ ]:
model.train()
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
max_steps = 5000 if GPU else 1000
hist = []
for step in range(max_steps):
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0 or step == max_steps - 1:
        model.eval()
        with torch.no_grad():
            _, vl = model(*get_batch("val"))
        hist.append((step, loss.item(), vl.item()))
        print(f"step {step:4d}  train {loss.item():.3f}  val {vl.item():.3f}")
        model.train()
# Expected T4 (5k): step 100 ~3.4, step 1000 ~2.0, step 5000 ~1.4 (val similar)
# Expected CPU (1k, small model): step 1000 train/val ~2.0–2.5
print(f"best val: {min(v for _,_,v in hist):.3f}  ->  perplexity {math.exp(min(v for _,_,v in hist)):.2f}")


In [ ]:
import matplotlib.pyplot as plt
steps, tr, va = zip(*hist)
plt.figure(figsize=(7, 4))
plt.plot(steps, tr, label="train"); plt.plot(steps, va, label="val")
plt.axhline(math.log(65), ls="--", label="ln 65 (random)")
plt.xlabel("step"); plt.ylabel("cross-entropy loss"); plt.legend()
plt.savefig("/tmp/loss.png")
plt.show()
# Expected: both curves dive from 4.17; val flattens ~1.4 (T4) or ~2.2 (CPU)


## 4. Sampling at three temperatures
Temperature divides logits before softmax. Watch the prose change.

In [ ]:
import time
@torch.no_grad()
def generate(prompt, max_new=500, temperature=1.0):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new):
        idx_in = idx[:, -model.block_size:]
        logits, _ = model(idx_in)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
    return decode(idx[0].tolist())

for T in (0.5, 1.0, 1.5):
    t0 = time.time()
    out = generate("KING:", temperature=T)
    dt = time.time() - t0
    print(f"--- T={T} ({500/dt:.0f} chars/s) ---")
    print(out[:400], "\n")
# Expected: T=0.5 repetitive thee/thou boilerplate; T=1.0 plausible Shakespeare with
# occasional nonsense words; T=1.5 Shakespearean-flavored gibberish


## 5. Temperature on paper: logits [3, 2, 1]
Verify the packet's worked example exactly.

In [ ]:
logits = torch.tensor([3.0, 2.0, 1.0])
for T in (0.5, 1.0, 1.5):
    p = F.softmax(logits / T, dim=-1)
    print(f"T={T}: " + " ".join(f"{x:.3f}" for x in p))
# Expected: T=0.5: 0.867 0.117 0.016 | T=1.0: 0.665 0.245 0.090 | T=1.5: 0.563 0.289 0.148


## 6. What to measure
| Metric | Your number |
|---|---|
| Total params (T4 expect 10,725,888) | |
| Step-0 loss (expect ≈ 4.17) | |
| Best val loss / perplexity | |
| Wall time for full training | |
| Generation speed, chars/s | |

**Checkpoints:** 1) Initial loss at vocab 65 → ln 65 ≈ 4.17. 2) Loss 1.4 → perplexity e^1.4 ≈ 4.06. 3) Tying the head saves vocab×d params: 24,960 here; in Llama-3-8B it would be 128k×4096 = 524M — usually not tied there.